In [ ]:
import os
import cv2
import numpy as np
import scipy.io as sio
import shutil
from pathlib import Path
from tqdm import tqdm

ROOT     = Path("..").resolve()
RAW      = ROOT / "data" / "raw"
PROC     = ROOT / "data" / "processed"

TRAIN_IMG  = RAW / "WIDER_train" / "images"
VAL_IMG    = RAW / "WIDER_val"   / "images"
ANNO_DIR   = RAW / "wider_face_split"

# Output folders (YOLO expects this structure)
for split in ["train", "val"]:
    (PROC / "images" / split).mkdir(parents=True, exist_ok=True)
    (PROC / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Output folders created:")
print(PROC)

In [ ]:
# === Config ===
MIN_FACE = 30
REMOVE_OUTSIDE_BOXES = True

def convert_wider_to_yolo(anno_path, img_root, out_img_dir, out_lbl_dir, split_name):
    anno = sio.loadmat(str(anno_path), squeeze_me=True, struct_as_record=False)
    
    event_list = anno["event_list"]
    file_list  = anno["file_list"]
    face_data  = anno["face_bbx_list"]

    images_saved   = 0
    faces_saved    = 0
    images_skipped = 0

    for i, event in enumerate(tqdm(event_list, desc=split_name)):
        files = file_list[i]
        for j in range(len(files)):
            img_path = img_root / event / (files[j] + ".jpg")
            
            if not img_path.exists():
                images_skipped += 1
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                images_skipped += 1
                continue
            
            ih, iw = img.shape[:2]

            boxes = face_data[i][j]
            if boxes.ndim == 1:
                boxes = boxes.reshape(1, -1)

            yolo_lines = []
            for box in boxes:
                x, y, w, h = box[:4].astype(float)

                if w < MIN_FACE or h < MIN_FACE or w <= 0 or h <= 0:
                    continue

                if REMOVE_OUTSIDE_BOXES:
                    if x < 0 or y < 0 or x + w > iw or y + h > ih:
                        continue

                cx = (x + w / 2) / iw
                cy = (y + h / 2) / ih
                nw = w / iw
                nh = h / ih

                cx = max(0, min(1, cx))
                cy = max(0, min(1, cy))
                nw = max(0, min(1, nw))
                nh = max(0, min(1, nh))

                yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

            if len(yolo_lines) == 0:
                continue

            shutil.copy2(str(img_path), str(out_img_dir / img_path.name))

            dest_lbl = out_lbl_dir / (img_path.stem + ".txt")
            dest_lbl.write_text("\n".join(yolo_lines))

            # ✅ Fixed counter (was bugged before)
            faces_saved  += len(yolo_lines)
            images_saved += 1

    print(f"\n{split_name} done:")
    print(f"  Images saved  : {images_saved:,}")
    print(f"  Faces saved   : {faces_saved:,}")
    print(f"  Images skipped: {images_skipped:,}")
    
    return images_saved, faces_saved


In [ ]:
# Quick sanity check on first annotation
anno_check = sio.loadmat(str(ANNO_DIR / "wider_face_train.mat"), 
                          squeeze_me=True, struct_as_record=False)
event_list_c = anno_check["event_list"]
file_list_c  = anno_check["file_list"]
face_data_c  = anno_check["face_bbx_list"]

print("Event :", event_list_c[0])
print("File  :", file_list_c[0][0])
print("Boxes :")
print(face_data_c[0][0])
print()
print("Box shape:", np.array(face_data_c[0][0]).shape)
print("Looks sensible?", all(face_data_c[0][0][:2] > 0))  # x,y should be positive

In [ ]:
train_imgs, train_faces = convert_wider_to_yolo(
    anno_path   = ANNO_DIR / "wider_face_train.mat",
    img_root    = TRAIN_IMG,
    out_img_dir = PROC / "images" / "train",
    out_lbl_dir = PROC / "labels" / "train",
    split_name  = "TRAIN"
)

val_imgs, val_faces = convert_wider_to_yolo(
    anno_path   = ANNO_DIR / "wider_face_val.mat",
    img_root    = VAL_IMG,
    out_img_dir = PROC / "images" / "val",
    out_lbl_dir = PROC / "labels" / "val",
    split_name  = "VAL"
)

print(f"\nFinal dataset:")
print(f"  Train: {train_imgs:,} images, {train_faces:,} faces")
print(f"  Val  : {val_imgs:,} images, {val_faces:,} faces")

In [ ]:
# Spot check — print first label file contents
first_label = next((PROC / "labels" / "train").glob("*.txt"))
print(f"File: {first_label.name}")
print(f"Contents:")
print(first_label.read_text())


In [ ]:
yaml_content = f"""# WIDER FACE - Face Detection Dataset
path: {str(PROC)}
train: images/train
val: images/val

nc: 1
names:
  0: face
"""

yaml_path = ROOT / "data" / "face_dataset.yaml"
yaml_path.write_text(yaml_content)
print("Dataset config saved:")
print(yaml_path)
print()
print(yaml_content)